# Lesson 5 — Outstanding Principal & Delinquency

**Project:** Loan Portfolio Performance & Risk Analytics  
**SQL:** DuckDB running inside Google Colab  
**Snapshot:** 31 August 2026 (end of day)  
**Currency:** IDR (Indonesian rupiah)  
**Data:** entirely synthetic

### Today's learning goals
1. Understand outstanding principal versus cumulative disbursement.
2. Calculate installment-level unpaid balance, days past due (DPD), and loan-level risk.
3. Calculate Portfolio at Risk based on loan **outstanding principal**.
4. Interpret lender and risk segment results without overstating conclusions.

> **Important:** This is an educational simplification, not a regulatory reporting formula. Real loan accounts have additional rules for restructures, reversals, write-offs, payment allocation, and NPL classifications. The data generator intentionally simulates weaker payment behavior in certain segments; do not infer real-world causality.


## Step 1 — Start DuckDB

Run this cell first. No database server installation is required. Colab needs an internet connection to install the DuckDB Python package.

In [ ]:
%pip -q install duckdb
import duckdb
con = duckdb.connect(database=":memory:")
print("DuckDB ready:", duckdb.__version__)

## Step 2 — Upload the five CSV files

Choose `borrowers.csv`, `lenders.csv`, `loans.csv`, `installments.csv`, and `repayments.csv` from **Lesson 3**. Run after the DuckDB setup cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
required = {"borrowers.csv", "lenders.csv", "loans.csv", "installments.csv", "repayments.csv"}
missing = required - set(uploaded)
if missing:
    raise ValueError(f"Please upload the missing file(s): {sorted(missing)}")
for name in ["borrowers", "lenders", "loans", "installments", "repayments"]:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{name}.csv')")
print("Five tables are ready!")


## Step 3 — Build the loan snapshot (three SQL views)

**Order matters:**
1. Aggregate repayments by installment.
2. Calculate how much principal/interest remains unpaid on each installment.
3. Roll these results up to one row per loan.

**DPD logic:** An installment is overdue only when its due date is **before** 31 Aug 2026 and any scheduled amount remains unpaid. Loan DPD is the maximum age (in days) across unpaid overdue installments, equivalent to counting from the **oldest** missed due date. Future installments are outstanding but not overdue.

**Avoid double-counting:** Do **not** sum loan principals after directly joining loans to multiple repayment rows.

### 1. Aggregate repayments per installment

In [ ]:
con.execute("""
-- STEP 1. Aggregate repayments FIRST: otherwise joining each loan to many
-- installments / payments can duplicate the loan principal.
CREATE OR REPLACE VIEW v_installment_payments AS
SELECT
    installment_id,
    SUM(CAST(principal_paid AS DECIMAL(18, 2))) AS principal_paid,
    SUM(CAST(interest_paid AS DECIMAL(18, 2))) AS interest_paid
FROM repayments
WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31'
GROUP BY installment_id;
""")
print("Created view v_installment_payments")

### 2. Calculate installment-level balances & DPD

In [ ]:
con.execute("""
-- STEP 2. Compare scheduled installment amounts with payments.
-- Future scheduled installments remain outstanding, but NOT overdue.
CREATE OR REPLACE VIEW v_installment_snapshot AS
WITH balances AS (
    SELECT
        i.installment_id,
        i.loan_id,
        CAST(i.due_date AS DATE) AS due_date,
        CAST(i.principal_due AS DECIMAL(18, 2)) AS principal_due,
        CAST(i.interest_due AS DECIMAL(18, 2)) AS interest_due,
        COALESCE(p.principal_paid, 0) AS principal_paid,
        COALESCE(p.interest_paid, 0) AS interest_paid,
        GREATEST(
            CAST(i.principal_due AS DECIMAL(18, 2)) - COALESCE(p.principal_paid, 0), 0
        ) AS remaining_principal,
        GREATEST(
            CAST(i.interest_due AS DECIMAL(18, 2)) - COALESCE(p.interest_paid, 0), 0
        ) AS remaining_interest
    FROM installments AS i
    LEFT JOIN v_installment_payments AS p
        ON i.installment_id = p.installment_id
)
SELECT
    *,
    CASE
        WHEN due_date < DATE '2026-08-31'
             AND remaining_principal + remaining_interest > 0
        THEN DATE_DIFF('day', due_date, DATE '2026-08-31')
        ELSE 0
    END AS installment_dpd,
    CASE
        WHEN due_date < DATE '2026-08-31'
        THEN remaining_principal
        ELSE 0
    END AS overdue_principal
FROM balances;
""")
print("Created view v_installment_snapshot")

### 3. Aggregate into one loan snapshot

In [ ]:
con.execute("""
-- STEP 3. Make ONE row per loan (safe to join to loan dimensions).
CREATE OR REPLACE VIEW v_loan_snapshot AS
WITH rollup AS (
    SELECT
        loan_id,
        SUM(remaining_principal) AS outstanding_principal,
        SUM(overdue_principal) AS overdue_principal,
        MAX(installment_dpd) AS dpd
    FROM v_installment_snapshot
    GROUP BY loan_id
)
SELECT
    l.loan_id,
    l.borrower_id,
    l.lender_id,
    CAST(l.disbursement_date AS DATE) AS disbursement_date,
    CAST(l.principal_amount AS DECIMAL(18, 2)) AS disbursed_principal,
    COALESCE(r.outstanding_principal, 0) AS outstanding_principal,
    COALESCE(r.overdue_principal, 0) AS overdue_principal,
    COALESCE(r.dpd, 0) AS dpd,
    CASE
        WHEN COALESCE(r.outstanding_principal, 0) = 0 THEN 'Paid off'
        WHEN COALESCE(r.dpd, 0) = 0 THEN 'Current'
        WHEN r.dpd BETWEEN 1 AND 30 THEN '1-30'
        WHEN r.dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN r.dpd BETWEEN 61 AND 89 THEN '61-89'
        ELSE '90+'
    END AS dpd_bucket
FROM loans AS l
LEFT JOIN rollup AS r ON l.loan_id = r.loan_id
WHERE CAST(l.disbursement_date AS DATE) <= DATE '2026-08-31';
""")
print("Created view v_loan_snapshot")

### Quick preview — one row per loan

Check `outstanding_principal`, `overdue_principal`, `dpd`, and `dpd_bucket`.

In [ ]:
con.sql("SELECT * FROM v_loan_snapshot ORDER BY loan_id LIMIT 10").df()

## Step 4 — Portfolio health: disbursement, outstanding & PAR

**PAR >30:** share of all outstanding principal associated with loans at DPD 31+. **PAR 90+:** share associated with loans at DPD 90 or more. These ratios are balance-weighted, not loan-count percentages.

In [ ]:
con.sql("""
-- QUESTION A: Overall portfolio health
-- PAR >30 = SUM(outstanding on loans with DPD > 30) / total outstanding
-- PAR 90+ = SUM(outstanding on loans with DPD >= 90) / total outstanding
SELECT
    COUNT(*) AS total_loans,
    COUNT(*) FILTER (WHERE outstanding_principal > 0) AS loans_with_outstanding,
    SUM(disbursed_principal) AS cumulative_disbursement_idr,
    SUM(outstanding_principal) AS outstanding_principal_idr,
    SUM(overdue_principal) AS overdue_principal_idr,
    COUNT(*) FILTER (WHERE dpd > 0) AS delinquent_loans,
    COUNT(*) FILTER (WHERE dpd > 30) AS loans_dpd_over_30,
    ROUND(
        100.0 * SUM(CASE WHEN dpd > 30 THEN outstanding_principal ELSE 0 END)
        / NULLIF(SUM(outstanding_principal), 0), 2
    ) AS par_over_30_pct,
    ROUND(
        100.0 * SUM(CASE WHEN dpd >= 90 THEN outstanding_principal ELSE 0 END)
        / NULLIF(SUM(outstanding_principal), 0), 2
    ) AS par_90_plus_pct
FROM v_loan_snapshot;
""").df()

### DPD bucket distribution

Paid off loans are separated from current, still-outstanding loans. Compare both **loan counts** and **outstanding balances**.

In [ ]:
con.sql("""
-- QUESTION B: What is the outstanding distribution across DPD buckets?
SELECT
    dpd_bucket,
    COUNT(*) AS loan_count,
    SUM(outstanding_principal) AS outstanding_principal_idr,
    SUM(overdue_principal) AS overdue_principal_idr,
    ROUND(
        100.0 * SUM(outstanding_principal)
        / NULLIF((SELECT SUM(outstanding_principal) FROM v_loan_snapshot), 0), 2
    ) AS share_of_total_outstanding_pct
FROM v_loan_snapshot
GROUP BY dpd_bucket
ORDER BY CASE dpd_bucket
    WHEN 'Paid off' THEN 0 WHEN 'Current' THEN 1
    WHEN '1-30' THEN 2 WHEN '31-60' THEN 3
    WHEN '61-89' THEN 4 ELSE 5 END;
""").df()

### Lender risk comparison

Compare cumulative disbursement, outstanding principal, and each lender’s own PAR >30 percentage. Larger disbursement does not necessarily imply worse repayment performance.

In [ ]:
con.sql("""
-- QUESTION C: Which lender has a higher share of delinquent outstanding?
SELECT
    ld.lender_name,
    COUNT(*) AS total_loans,
    SUM(s.disbursed_principal) AS cumulative_disbursement_idr,
    SUM(s.outstanding_principal) AS outstanding_principal_idr,
    ROUND(
        100.0 * SUM(CASE WHEN s.dpd > 30 THEN s.outstanding_principal ELSE 0 END)
        / NULLIF(SUM(s.outstanding_principal), 0), 2
    ) AS par_over_30_pct
FROM v_loan_snapshot AS s
JOIN lenders AS ld ON s.lender_id = ld.lender_id
GROUP BY ld.lender_name
ORDER BY par_over_30_pct DESC;
""").df()

### Borrower risk segment

Risk segments in this simulation are assigned during data generation, with deliberately different payment probabilities. Interpret these results only as a demonstration of the method.

In [ ]:
con.sql("""
-- QUESTION D: Which predefined risk segments have the highest PAR >30?
SELECT
    b.risk_segment,
    COUNT(*) AS total_loans,
    SUM(s.outstanding_principal) AS outstanding_principal_idr,
    ROUND(
        100.0 * SUM(CASE WHEN s.dpd > 30 THEN s.outstanding_principal ELSE 0 END)
        / NULLIF(SUM(s.outstanding_principal), 0), 2
    ) AS par_over_30_pct
FROM v_loan_snapshot AS s
JOIN borrowers AS b ON s.borrower_id = b.borrower_id
GROUP BY b.risk_segment
ORDER BY par_over_30_pct DESC;
""").df()

### Check impossible installment balances

Both results should be zero. Check for overpayments before publishing dashboards.

In [ ]:
con.sql("""
-- QUALITY CHECK A: no overpaid installments / negative unpaid balances.
SELECT
    COUNT(*) FILTER (WHERE remaining_principal < 0 OR remaining_interest < 0)
        AS negative_installment_balances,
    COUNT(*) FILTER (WHERE principal_paid > principal_due OR interest_paid > interest_due)
        AS overpaid_installments
FROM v_installment_snapshot;
""").df()

### Reconcile portfolio principal

`reconciliation_difference_idr` should equal exactly **0.00**.

In [ ]:
con.sql("""
-- QUALITY CHECK B: total disbursement less total principal paid
-- must equal outstanding from the installment schedule (to within IDR 0.01).
SELECT
    SUM(s.disbursed_principal) AS disbursed_idr,
    (SELECT SUM(CAST(principal_paid AS DECIMAL(18,2)))
       FROM repayments WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31')
        AS principal_paid_idr,
    SUM(s.outstanding_principal) AS outstanding_idr,
    SUM(s.disbursed_principal) -
    (SELECT SUM(CAST(principal_paid AS DECIMAL(18,2)))
       FROM repayments WHERE CAST(payment_date AS DATE) <= DATE '2026-08-31') -
    SUM(s.outstanding_principal) AS reconciliation_difference_idr
FROM v_loan_snapshot AS s;
""").df()

## Reference results for the provided synthetic dataset

- Cumulative disbursement: approximately **IDR 31.704 billion**.
- Outstanding principal: approximately **IDR 9.593 billion**.
- Outstanding on loans with DPD >30: approximately **59.98%** of total outstanding.
- Outstanding on loans with DPD ≥90: approximately **48.89%** of total outstanding.
- Highest cumulative disbursement lender: **Meridian Capital**.
- Highest PAR >30 lender: **Harbor Finance**.

These are **simulation results**, not evidence about any real lender or lending market. The generator intentionally produces severe delinquency scenarios to make the analytical exercise meaningful.


## Your mini-assignment (answer in English)

1. What is the difference between cumulative disbursement and outstanding principal?
2. Which lender has the highest PAR >30 and what is its figure?
3. Why should we not conclude that the lender *caused* higher delinquency from this analysis alone?
4. What is one follow-up analysis you would recommend to the management team?

**Example (not your final answer):** "Although Meridian Capital has the largest disbursed portfolio, lender-level risk should be assessed using outstanding-based delinquency rates rather than loan volume alone."

### Publish to GitHub
1. Upload `02_outstanding_and_delinquency.sql` to your `sql/` directory.
2. Save this notebook to a `.ipynb` file and upload it to `notebooks/`.
3. In your main README, tick the SQL analysis item when you are satisfied with the results.
4. Do not publish credentials, private financial/customer data, or unsupported real-world conclusions.
